In [ ]:
import datetime as dt
from pathlib import Path
from typing import cast
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.colors import ListedColormap
from matplotlib.figure import Figure
from matplotlib.lines import Line2D

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config, resolve_data_dir
from eddy_tracking.preprocess.streamline import KM_PER_DEG_LAT, compute_signed_distance_km, trace_streamline_for_file
from eddy_tracking.preprocess.swot import index_swot_files_by_date
from eddy_tracking.preprocess.tracks import load_track_observations

EXPERIMENT = 'gulf_stream_20240305_20260531'
AXIS_DATE = dt.date(2025, 10, 5)
ARROW_STRIDE = 8
ARROW_AXIS_KM = 75
NEAR_AXIS_KM = 150
FIGURE_DIR = Path('/Users/jerry/school/research/research_paper/figures')
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
identity_columns = ['polarity', 'track_id']

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

In [ ]:
swot_file = index_swot_files_by_date(resolve_data_dir(cfg, 'swot_dir'))[AXIS_DATE]
axis = trace_streamline_for_file(swot_file, tuple(cfg['gulf_stream']['lat_band']))
with xr.open_dataset(swot_file) as ds:
    ds = ds.isel(time=0)
    lon = ds['longitude'].to_numpy()
    lat = ds['latitude'].to_numpy()
    ugos = ds['ugos'].to_numpy()
    vgos = ds['vgos'].to_numpy()
speed = np.hypot(ugos, vgos)
half = ARROW_STRIDE // 2
arrow_lon, arrow_lat = np.meshgrid(lon[half::ARROW_STRIDE], lat[half::ARROW_STRIDE])
arrow_u = ugos[half::ARROW_STRIDE, half::ARROW_STRIDE] / np.cos(np.radians(arrow_lat))
arrow_v = vgos[half::ARROW_STRIDE, half::ARROW_STRIDE]
arrow_length = np.hypot(arrow_u, arrow_v)
near_axis = np.array([
    abs(compute_signed_distance_km(axis.lon, axis.lat, point_lon, point_lat)[0]) <= ARROW_AXIS_KM
    for point_lon, point_lat in zip(arrow_lon.ravel(), arrow_lat.ravel())
]).reshape(arrow_lon.shape) & np.isfinite(arrow_length)
band_lon, band_lat = np.meshgrid(lon[::2], lat[::2])
axis_distance_km = np.array([
    abs(compute_signed_distance_km(axis.lon, axis.lat, point_lon, point_lat)[0])
    for point_lon, point_lat in zip(band_lon.ravel(), band_lat.ravel())
]).reshape(band_lon.shape)
axis_distance_km[np.isnan(speed[::2, ::2])] = np.nan

fig, ax = cast(tuple[Figure, GeoAxes], plt.subplots(figsize=(6.69, 3.9), subplot_kw={'projection': ccrs.PlateCarree()}, layout='constrained'))
ax.set_extent([*cfg['base']['region']['lon_range'], *cfg['base']['region']['lat_range']], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=0)
ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
grid = ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=2)
grid.top_labels = False
grid.right_labels = False
grid.xlabel_style = {'size': 8}
grid.ylabel_style = {'size': 8}
speed_cmap = ListedColormap(plt.get_cmap('Blues')(np.linspace(0.12, 1, 256)))
mesh = ax.pcolormesh(lon, lat, speed, cmap=speed_cmap, vmin=0, vmax=2.5, shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
ax.quiver(
    arrow_lon[near_axis], arrow_lat[near_axis], (arrow_u / arrow_length)[near_axis], (arrow_v / arrow_length)[near_axis],
    color='#222222', edgecolor='white', linewidth=0.5, scale=1 / 0.55, scale_units='xy', angles='xy', width=0.0028, headwidth=3.5, headlength=4.5, headaxislength=4,
    transform=ccrs.PlateCarree(), zorder=4,
)
ax.contour(band_lon, band_lat, axis_distance_km, levels=[NEAR_AXIS_KM], colors=['#222222'], linestyles=':', linewidths=0.8, transform=ccrs.PlateCarree(), zorder=5)
ax.plot(axis.lon, axis.lat, color='#e6550d', linewidth=1.0, transform=ccrs.PlateCarree(), zorder=5)
colorbar = fig.colorbar(mesh, cax=ax.inset_axes((1.02, 0, 0.025, 1)))
colorbar.set_ticks([0, 0.5, 1, 1.5, 2, 2.5])
colorbar.set_label('Geostrophic speed (m s$^{-1}$)')
handles = [
    Line2D([], [], color='#e6550d', linewidth=1.0, label='Gulf Stream axis'),
    Line2D([], [], color='#222222', linewidth=0.8, linestyle=':', label=f'{NEAR_AXIS_KM} km from the axis'),
]
ax.legend(handles=handles, loc='lower right', frameon=True, framealpha=0.95, edgecolor='none')
ax.set_title(f'{AXIS_DATE:%Y-%m-%d}', loc='left')
FIGURE_DIR.mkdir(exist_ok=True)
fig.savefig(FIGURE_DIR / 'gulf_stream_axis.pdf')
plt.show()

In [ ]:
movement = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
target_class = cast(pd.Series, movement['polarity']).map(target_classes)
movement['crossed_axis'] = movement['movement'].eq(target_class)
movement['near_axis_birth'] = movement['birth_distance_km'].abs().le(NEAR_AXIS_KM) & movement['death_side'].eq(target_class.str[1])
movement['is_target'] = movement['crossed_axis'] | movement['near_axis_birth']
sensitivity_rows = []
for polarity in polarity_names:
    rows = movement.loc[movement['polarity'].eq(polarity)]
    on_target_side = rows['death_side'].eq(target_classes[polarity][1])
    for threshold in (50, 100, 150, 200, 250, 300):
        near_only = rows['birth_distance_km'].abs().le(threshold) & on_target_side & ~rows['crossed_axis']
        sensitivity_rows.append({'polarity': polarity, 'threshold_km': threshold, 'crossed': int(rows['crossed_axis'].sum()), 'near_birth_only': int(near_only.sum())})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity['target'] = sensitivity['crossed'] + sensitivity['near_birth_only']
print(f'Target eddies per polarity as the near-axis threshold changes; the crossing rule does not depend on it, and {NEAR_AXIS_KM} km is the value in use.')
display(sensitivity.pivot(index='threshold_km', columns='polarity', values=['crossed', 'near_birth_only', 'target']).reindex(columns=pd.MultiIndex.from_product([['crossed', 'near_birth_only', 'target'], polarity_names])))

track_radius = load_track_observations(EXPERIMENT).groupby(identity_columns)['radius_km'].mean()
target_radius = movement.loc[movement['is_target']].merge(track_radius, on=identity_columns)['radius_km']
streamline = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet')
streamline['lon_bin'] = ((streamline['lon'] - cfg['base']['region']['lon_range'][0]) // 0.25).astype(int)
daily_axis_lat = streamline.groupby(['date', 'lon_bin'])['lat'].mean().unstack('lon_bin')
daily_axis_lat = daily_axis_lat.loc[:, daily_axis_lat.notna().sum() >= len(daily_axis_lat) / 2]
axis_spread_km = np.diff(np.nanpercentile(daily_axis_lat.to_numpy(), [10, 90], axis=0), axis=0)[0] * KM_PER_DEG_LAT
daily_shift_km = daily_axis_lat.diff().abs().to_numpy() * KM_PER_DEG_LAT
scales = pd.DataFrame({
    'scale': ['target eddy speed radius', 'all track speed radius', 'daily axis 10th to 90th percentile spread', 'axis shift between consecutive days'],
    'median_km': [target_radius.median(), track_radius.median(), np.median(axis_spread_km), np.nanmedian(daily_shift_km)],
    'p90_km': [target_radius.quantile(0.9), track_radius.quantile(0.9), np.percentile(axis_spread_km, 90), np.nanpercentile(daily_shift_km, 90)],
}).round(0)
print(f'Length scales behind the {NEAR_AXIS_KM} km threshold: the speed radius per track, the north-south spread of the daily axis at each longitude across all days, and its shift from one day to the next.')
display(scales)

In [ ]:
with xr.open_dataset(swot_file) as ds:
    sla = ds['sla'].isel(time=0).to_numpy()

sla_fig, sla_ax = cast(tuple[Figure, GeoAxes], plt.subplots(figsize=(6.69, 3.9), subplot_kw={'projection': ccrs.PlateCarree()}, layout='constrained'))
sla_ax.set_extent([*cfg['base']['region']['lon_range'], *cfg['base']['region']['lat_range']], crs=ccrs.PlateCarree())
sla_ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=0)
sla_ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
sla_grid = sla_ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2), zorder=2)
sla_grid.top_labels = False
sla_grid.right_labels = False
sla_grid.xlabel_style = {'size': 8}
sla_grid.ylabel_style = {'size': 8}
sla_mesh = sla_ax.pcolormesh(lon, lat, sla, cmap='RdBu_r', vmin=-1, vmax=1, shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
sla_ax.plot(axis.lon, axis.lat, color='#111111', linewidth=1.0, transform=ccrs.PlateCarree(), zorder=5, label='Gulf Stream axis')
sla_colorbar = sla_fig.colorbar(sla_mesh, cax=sla_ax.inset_axes((1.02, 0, 0.025, 1)))
sla_colorbar.set_ticks([-1, -0.5, 0, 0.5, 1])
sla_colorbar.set_label('SLA (m)')
sla_ax.legend(loc='lower right', frameon=True, framealpha=0.95, edgecolor='none')
sla_ax.set_title(f'{AXIS_DATE:%Y-%m-%d}', loc='left')
sla_fig.savefig(FIGURE_DIR / 'gulf_stream_sla.pdf')
plt.show()